# Testing preflightml on a churn dataset

This notebook loads `churn_data.csv` (a synthetic but realistic telecom churn
dataset - same shape as the well-known Telco Churn dataset), splits it into
train/test, and runs `preflightml` on it.

Unlike `demo.py` (which used a deliberately broken dataset to show every check
firing at once), this data is *mostly* clean - which is the more realistic
day-to-day case. See what it catches anyway.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from preflightml import inspect

df = pd.read_csv("churn_data.csv", parse_dates=["signup_date"])
df.shape

## Look at the data first

Always worth a quick look before running any automated check - you want your
own eyes on the data too, not just the tool's.

In [ ]:
df.head()

In [ ]:
df["Churn"].value_counts(normalize=True)

## Split into train/test

We split *before* running `inspect()` so the library can check split-integrity
issues (row overlap, entity overlap, distribution shift) - those checks only
run when you give it a `test=` frame.

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=1, stratify=df["Churn"])
train.shape, test.shape

## Run the inspection

`time_col` and `entity_col` are passed explicitly rather than left to be
guessed - remember, guessing lowers confidence on any related finding, so
it's better to state them when you know them.

In [ ]:
report = inspect(
    train,
    target="Churn",
    test=test,
    time_col="signup_date",
    entity_col="customerID",
)
report.summary()

## See the full detail on anything Critical or Warning

In [ ]:
report.details("critical")

In [ ]:
report.details("warning")

## Drill into specific columns

This is where the library's actual differentiator shows up - not just "there's
an issue" but exactly which column, which statistic, and why.

In [ ]:
report.explain("CancellationTicketID")

In [ ]:
report.explain("TotalCharges")

## Export a shareable HTML report

In [ ]:
report.to_html("churn_report.html")
print("saved churn_report.html - open it in a browser")

## CI/CD style gate check

In [ ]:
if report.has_blocking_issues:
    print("BLOCKED: fix critical issues before training a final model")
else:
    print("Clear to train")